# E-Commerce KNN: Visualisierungen

In diesem Notebook zeigen wir, wie man die Ergebnisse eines KNNs visuell aufbereitet, um sie z.B. dem Management oder in Prüfungen optimal zu präsentieren.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# 1. Daten laden und vorbereiten (Wie im fortgeschrittenen Notebook)
df = pd.read_csv("online_shoppers_intention.csv")
df_encoded = pd.get_dummies(df, columns=['Month', 'VisitorType', 'Weekend'], drop_first=True)

X = df_encoded.drop(columns=['Revenue'])
y = df_encoded['Revenue'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. KNN Trainieren (ohne Turnier, damit es für die Diagramme schnell geht)
print("Trainiere das KNN... Bitte warten.")
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', max_iter=500, random_state=42)
mlp.fit(X_train_scaled, y_train)
print("Training abgeschlossen!")

### 1. Die Lernkurve (Loss Curve)
Zeigt, wie der Fehler der KI über die Epochen hinweg sinkt.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, color='blue', linewidth=2)
plt.title("Lernkurve der KI (Loss Curve)", fontsize=14)
plt.xlabel("Epochen", fontsize=12)
plt.ylabel("Fehler (Loss)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### 2. Confusion Matrix Heatmap
Eine farbliche Darstellung der wahren vs. falschen Vorhersagen.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay.from_estimator(
    mlp, X_test_scaled, y_test, 
    display_labels=['Kein Kauf (0)', 'Kauf (1)'], 
    cmap='Blues', ax=ax
)
plt.title("Confusion Matrix Heatmap", fontsize=14)
plt.show()

### 3. ROC Kurve
Wichtig für Prüfungen: Zeigt das Verhältnis zwischen True Positive Rate und False Positive Rate.

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(mlp, X_test_scaled, y_test, ax=ax, color='darkorange')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Zufalls-Raten')
plt.title("ROC Kurve", fontsize=14)
plt.legend()
plt.show()

### 4. Permutation Feature Importance
Welche Daten haben die Entscheidung der KI am meisten beeinflusst?

In [ ]:
from sklearn.inspection import permutation_importance

print("Berechne Wichtigkeiten... (kann kurz dauern)")
ergebnis = permutation_importance(mlp, X_test_scaled, y_test, n_repeats=5, random_state=42, n_jobs=-1)

# Wir nehmen nur die Top 10 wichtigsten Features für eine saubere Grafik
wichtigkeiten = pd.Series(ergebnis.importances_mean, index=X.columns)
top_10 = wichtigkeiten.sort_values().tail(10)

plt.figure(figsize=(10, 6))
top_10.plot(kind='barh', color='teal')
plt.title("Top 10 wichtigste Faktoren für die Kauf-Entscheidung", fontsize=14)
plt.xlabel("Wichtigkeit (Drop in Accuracy)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()